# OctoTetrahedral AGI -- ARC Grid Training v2
**Direct grid-to-grid transformer | Per-cell classification | Data augmentation**

This notebook trains a standalone grid transformer on ARC-AGI tasks.
It encodes grids as 10-color embeddings (not text tokens) and predicts each output cell independently.

**Runtime**: GPU recommended (T4 free tier works). CPU works but is ~10x slower.

**Run all cells** -- takes ~15 min on T4 GPU for 2000 steps.

In [ ]:
# CELL 1: Clone repo + install deps + download ARC data
!rm -rf /content/octo
!git clone --depth 1 https://github.com/GitMonsters/octotetrahedral-agi.git /content/octo
%cd /content/octo
!pip install -q torch numpy tqdm

# Download ARC-AGI dataset
!rm -rf arc_data
!git clone --depth 1 https://github.com/fchollet/ARC-AGI.git arc_data 2>/dev/null

import glob, os
train_files = sorted(glob.glob('arc_data/data/training/*.json'))
eval_files = sorted(glob.glob('arc_data/data/evaluation/*.json'))
print(f'ARC-AGI: {len(train_files)} train + {len(eval_files)} eval tasks')
assert len(train_files) >= 400, 'ARC data download failed'
print('Setup complete!')

In [ ]:
# CELL 2: Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({mem:.0f} GB)')
else:
    print('No GPU -- enable via Runtime > Change runtime type > T4')
    print('Training will work on CPU but be slower.')
print(f'PyTorch {torch.__version__} | Device: {device}')

In [ ]:
# CELL 3: Grid utilities + Dataset
import json, math, time, random, logging
from pathlib import Path
from typing import List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger(__name__)

NUM_COLORS = 11  # 0-9 = ARC colors, 10 = PAD
PAD = 10
MAX_GRID = 15    # Most ARC grids fit in 15x15
MAX_EXAMPLES = 3

def pad_grid(grid, h, w):
    t = torch.full((h, w), PAD, dtype=torch.long)
    gh, gw = len(grid), len(grid[0]) if grid else 0
    for r in range(min(gh, h)):
        for c in range(min(gw, w)):
            t[r, c] = grid[r][c]
    return t

def augment_grid_pair(inp, out, rng):
    inp_arr = np.array(inp)
    out_arr = np.array(out)
    k = rng.randint(0, 3)
    if k > 0:
        inp_arr = np.rot90(inp_arr, k)
        out_arr = np.rot90(out_arr, k)
    if rng.random() < 0.5:
        inp_arr = np.fliplr(inp_arr)
        out_arr = np.fliplr(out_arr)
    if rng.random() < 0.5:
        inp_arr = np.flipud(inp_arr)
        out_arr = np.flipud(out_arr)
    if rng.random() < 0.5:
        perm = list(range(10))
        non_zero = perm[1:]
        rng.shuffle(non_zero)
        perm[1:] = non_zero
        inp_arr = np.vectorize(lambda x: perm[x])(inp_arr)
        out_arr = np.vectorize(lambda x: perm[x])(out_arr)
    return inp_arr.tolist(), out_arr.tolist()

def task_difficulty(data):
    total_cells = 0
    colors = set()
    for ex in data['train']:
        for grid in [ex['input'], ex['output']]:
            h, w = len(grid), len(grid[0]) if grid else 0
            total_cells += h * w
            for row in grid:
                colors.update(row)
    return total_cells + len(colors) * 10

class ARCGridDataset(Dataset):
    def __init__(self, data_dir, split='training', max_examples=MAX_EXAMPLES,
                 augment=True, curriculum=False, seed=42):
        self.max_examples = max_examples
        self.augment = augment
        self.rng = random.Random(seed)
        task_dir = Path(data_dir) / split
        self.tasks = []
        for f in sorted(task_dir.glob('*.json')):
            with open(f) as fh:
                data = json.load(fh)
            self.tasks.append((f.stem, data))
        if curriculum:
            self.tasks.sort(key=lambda t: task_difficulty(t[1]))
        self.samples = []
        for i, (tid, data) in enumerate(self.tasks):
            for j in range(len(data['test'])):
                self.samples.append((i, j))
        print(f'Loaded {len(self.tasks)} tasks, {len(self.samples)} samples ({split})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        task_idx, test_idx = self.samples[idx]
        tid, data = self.tasks[task_idx]
        train_exs = data['train'][:self.max_examples]
        test_pair = data['test'][test_idx]
        if self.augment:
            augmented = []
            for ex in train_exs:
                ai, ao = augment_grid_pair(ex['input'], ex['output'], self.rng)
                augmented.append({'input': ai, 'output': ao})
            ti, to = augment_grid_pair(
                test_pair['input'], test_pair.get('output', test_pair['input']), self.rng)
            train_exs = augmented
            test_input, test_output = ti, to
        else:
            test_input = test_pair['input']
            test_output = test_pair.get('output', test_pair['input'])
        context_grids = []
        for ex in train_exs:
            context_grids.append(pad_grid(ex['input'], MAX_GRID, MAX_GRID))
            context_grids.append(pad_grid(ex['output'], MAX_GRID, MAX_GRID))
        while len(context_grids) < self.max_examples * 2:
            context_grids.append(torch.full((MAX_GRID, MAX_GRID), PAD, dtype=torch.long))
        context = torch.stack(context_grids)
        test_in = pad_grid(test_input, MAX_GRID, MAX_GRID)
        test_out = pad_grid(test_output, MAX_GRID, MAX_GRID)
        out_h = len(test_output)
        out_w = len(test_output[0]) if test_output else 0
        return {'task_id': tid, 'context': context, 'test_input': test_in,
                'test_output': test_out, 'out_h': out_h, 'out_w': out_w}

def collate_fn(batch):
    return {
        'task_id': [b['task_id'] for b in batch],
        'context': torch.stack([b['context'] for b in batch]),
        'test_input': torch.stack([b['test_input'] for b in batch]),
        'test_output': torch.stack([b['test_output'] for b in batch]),
        'out_h': torch.tensor([b['out_h'] for b in batch]),
        'out_w': torch.tensor([b['out_w'] for b in batch]),
    }

print('Dataset classes ready!')

In [ ]:
# CELL 4: ARC Grid Transformer Model
class ARCGridModel(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=6,
                 dim_feedforward=1024, dropout=0.1,
                 max_grids=MAX_EXAMPLES * 2 + 1, grid_size=MAX_GRID):
        super().__init__()
        self.d_model = d_model
        self.grid_size = grid_size
        self.max_grids = max_grids
        self.color_embed = nn.Embedding(NUM_COLORS, d_model)
        self.row_embed = nn.Embedding(grid_size, d_model)
        self.col_embed = nn.Embedding(grid_size, d_model)
        self.grid_embed = nn.Embedding(max_grids + 1, d_model)
        self.role_embed = nn.Embedding(4, d_model)  # ctx_in, ctx_out, test_in, test_out_query
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Linear(dim_feedforward, 10))
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

    def encode_grid(self, grid, grid_idx, role):
        B, H, W = grid.shape
        device = grid.device
        tokens = self.color_embed(grid)
        rows = torch.arange(H, device=device).unsqueeze(1).expand(H, W)
        cols = torch.arange(W, device=device).unsqueeze(0).expand(H, W)
        tokens = tokens + self.row_embed(rows) + self.col_embed(cols)
        tokens = tokens + self.grid_embed(torch.tensor(grid_idx, device=device))
        tokens = tokens + self.role_embed(torch.tensor(role, device=device))
        return tokens.view(B, H * W, self.d_model)

    def forward(self, context, test_input, test_output=None, out_h=None, out_w=None):
        B = context.shape[0]
        num_ctx = context.shape[1]
        device = context.device
        all_tokens = []
        for i in range(num_ctx):
            role = 0 if i % 2 == 0 else 1
            all_tokens.append(self.encode_grid(context[:, i], grid_idx=i, role=role))
        all_tokens.append(self.encode_grid(test_input, grid_idx=num_ctx, role=2))
        query_grid = torch.full((B, MAX_GRID, MAX_GRID), PAD, dtype=torch.long, device=device)
        all_tokens.append(self.encode_grid(query_grid, grid_idx=num_ctx + 1, role=3))
        x = torch.cat(all_tokens, dim=1)
        x = self.transformer(x)
        query_out = x[:, -MAX_GRID * MAX_GRID:, :]
        query_out = query_out.view(B, MAX_GRID, MAX_GRID, self.d_model)
        logits = self.output_head(query_out)
        result = {'logits': logits}
        if test_output is not None:
            mask = test_output != PAD
            logits_flat = logits.view(-1, 10)
            targets_flat = test_output.view(-1)
            mask_flat = mask.view(-1)
            targets_clamped = targets_flat.clamp(0, 9)
            if mask_flat.any():
                loss = F.cross_entropy(logits_flat[mask_flat], targets_clamped[mask_flat])
            else:
                loss = torch.tensor(0.0, device=device)
            preds = logits.argmax(dim=-1)
            correct = ((preds == test_output) & mask).float().sum()
            total = mask.float().sum()
            cell_acc = correct / total.clamp(min=1)
            exact = 0
            for b in range(B):
                if out_h is not None and out_w is not None:
                    h, w = out_h[b].item(), out_w[b].item()
                    if (preds[b, :h, :w] == test_output[b, :h, :w]).all():
                        exact += 1
                else:
                    b_mask = mask[b]
                    if b_mask.any() and ((preds[b] == test_output[b]) | ~b_mask).all():
                        exact += 1
            result['loss'] = loss
            result['cell_accuracy'] = cell_acc.item()
            result['exact_match'] = exact / B
        return result

# Quick test
test_model = ARCGridModel(d_model=64, nhead=4, num_layers=2, dim_feedforward=256)
print(f'Model test: {test_model.get_num_params():,} params (tiny config)')
del test_model
print('Model class ready!')

In [ ]:
# CELL 5: Configure training
# Adjust these based on your GPU/time budget:

# === TRAINING CONFIG ===
DATA_DIR = 'arc_data/data'    # ARC-AGI data path
D_MODEL = 256                 # Model dimension (256=fast, 512=better)
NUM_LAYERS = 6                # Transformer layers
NHEAD = 8                     # Attention heads
BATCH_SIZE = 4                # Batch size (reduce if OOM)
MAX_STEPS = 2000              # Training steps (2000=~15min on T4)
LR = 3e-4                    # Learning rate
AUGMENT = True                # Data augmentation (rotation/flip/color perm)
CURRICULUM = True             # Easy-to-hard ordering
# ========================

print(f'Config: d={D_MODEL}, layers={NUM_LAYERS}, heads={NHEAD}')
print(f'Training: {MAX_STEPS} steps, batch={BATCH_SIZE}, lr={LR}')
print(f'Augmentation: {AUGMENT} | Curriculum: {CURRICULUM}')

In [ ]:
# CELL 6: Load ARC datasets
train_ds = ARCGridDataset(
    DATA_DIR, split='training',
    augment=AUGMENT, curriculum=CURRICULUM)
val_ds = ARCGridDataset(
    DATA_DIR, split='evaluation',
    augment=False, curriculum=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn, num_workers=2)

# Quick sanity check
batch = next(iter(train_loader))
print(f'Context shape: {batch["context"].shape}')
print(f'Test input shape: {batch["test_input"].shape}')
print(f'Test output shape: {batch["test_output"].shape}')
print(f'Sample task: {batch["task_id"][0]}')

In [ ]:
# CELL 7: Create model + Train!
model = ARCGridModel(
    d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS,
    dim_feedforward=D_MODEL * 4)
model = model.to(device)
n_params = model.get_num_params()
print(f'Model: {n_params:,} parameters on {device}')

# Optimizer + scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=MAX_STEPS, eta_min=LR * 0.01)

# Training loop
best_val_acc = 0.0
train_losses = []
val_accs = []
step = 0
running_loss = 0.0
running_acc = 0.0
running_exact = 0.0
start_time = time.time()

model.train()
print(f'Training {MAX_STEPS} steps...')
print('-' * 70)

while step < MAX_STEPS:
    for batch in train_loader:
        if step >= MAX_STEPS:
            break

        # Warmup (first 100 steps)
        if step < 100:
            lr_scale = (step + 1) / 100
            for pg in optimizer.param_groups:
                pg['lr'] = LR * lr_scale

        ctx = batch['context'].to(device)
        t_in = batch['test_input'].to(device)
        t_out = batch['test_output'].to(device)
        o_h = batch['out_h'].to(device)
        o_w = batch['out_w'].to(device)

        result = model(ctx, t_in, t_out, o_h, o_w)
        loss = result['loss']

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad()
            step += 1
            continue

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        running_acc += result['cell_accuracy']
        running_exact += result['exact_match']
        step += 1

        # Log every 50 steps
        if step % 50 == 0:
            avg_loss = running_loss / 50
            avg_acc = running_acc / 50
            avg_exact = running_exact / 50
            elapsed = time.time() - start_time
            eta = elapsed / step * (MAX_STEPS - step)
            print(f'Step {step:>5}/{MAX_STEPS} | Loss: {avg_loss:.4f} | '
                  f'Cell Acc: {avg_acc:.1%} | Exact: {avg_exact:.1%} | '
                  f'ETA: {eta/60:.0f}m')
            train_losses.append(avg_loss)
            running_loss = 0.0
            running_acc = 0.0
            running_exact = 0.0

        # Validate every 200 steps
        if step % 200 == 0:
            model.eval()
            val_loss = val_acc = val_exact = 0.0
            n_val = 0
            with torch.no_grad():
                for vb in val_loader:
                    if n_val >= 50:
                        break
                    vc = vb['context'].to(device)
                    vi = vb['test_input'].to(device)
                    vo = vb['test_output'].to(device)
                    vh = vb['out_h'].to(device)
                    vw = vb['out_w'].to(device)
                    vr = model(vc, vi, vo, vh, vw)
                    val_loss += vr['loss'].item()
                    val_acc += vr['cell_accuracy']
                    val_exact += vr['exact_match']
                    n_val += 1
            va = val_acc / max(n_val, 1)
            vl = val_loss / max(n_val, 1)
            ve = val_exact / max(n_val, 1)
            val_accs.append(va)
            tag = ''
            if va > best_val_acc:
                best_val_acc = va
                torch.save(model.state_dict(), 'best_arc_grid.pt')
                tag = ' ** NEW BEST **'
            print(f'  VAL | Loss: {vl:.4f} | Cell Acc: {va:.1%} | Exact: {ve:.1%}{tag}')
            model.train()

# Final save
torch.save(model.state_dict(), 'final_arc_grid.pt')
elapsed = time.time() - start_time
print('=' * 70)
print(f'Training complete! {MAX_STEPS} steps in {elapsed/60:.1f} min')
print(f'Best validation cell accuracy: {best_val_acc:.1%}')
print(f'Checkpoints: best_arc_grid.pt, final_arc_grid.pt')

In [ ]:
# CELL 8: Full evaluation on held-out tasks
# Load best checkpoint
model.load_state_dict(torch.load('best_arc_grid.pt', map_location=device))
model.eval()

total_cell_acc = 0.0
total_exact = 0.0
n_eval = 0
task_results = {}

with torch.no_grad():
    for batch in val_loader:
        ctx = batch['context'].to(device)
        t_in = batch['test_input'].to(device)
        t_out = batch['test_output'].to(device)
        o_h = batch['out_h'].to(device)
        o_w = batch['out_w'].to(device)
        result = model(ctx, t_in, t_out, o_h, o_w)
        total_cell_acc += result['cell_accuracy']
        total_exact += result['exact_match']
        for tid in batch['task_id']:
            task_results[tid] = task_results.get(tid, 0) + result['exact_match']
        n_eval += 1

avg_cell = total_cell_acc / max(n_eval, 1)
avg_exact = total_exact / max(n_eval, 1)
solved = sum(1 for v in task_results.values() if v > 0.5)

print('=' * 70)
print(f'EVALUATION RESULTS ({n_eval} batches, {len(task_results)} tasks)')
print(f'  Cell Accuracy:  {avg_cell:.1%}')
print(f'  Exact Match:    {avg_exact:.1%}')
print(f'  Tasks Solved:   {solved}/{len(task_results)}')
print('=' * 70)

In [ ]:
# CELL 9: Visualize predictions on random tasks
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

ARC_COLORS = [
    '#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
    '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'
]
cmap = mcolors.ListedColormap(ARC_COLORS)
norm = mcolors.BoundaryNorm(range(11), 10)

model.eval()
fig, axes = plt.subplots(4, 3, figsize=(12, 16))

# Get 4 random eval samples
indices = random.sample(range(len(val_ds)), 4)
for row, idx in enumerate(indices):
    sample = val_ds[idx]
    ctx = sample['context'].unsqueeze(0).to(device)
    t_in = sample['test_input'].unsqueeze(0).to(device)
    t_out = sample['test_output'].unsqueeze(0).to(device)
    oh = torch.tensor([sample['out_h']]).to(device)
    ow = torch.tensor([sample['out_w']]).to(device)

    with torch.no_grad():
        result = model(ctx, t_in, t_out, oh, ow)
    pred = result['logits'].argmax(dim=-1)[0].cpu().numpy()
    h, w = sample['out_h'], sample['out_w']

    # Input
    inp = sample['test_input'].numpy()
    inp[inp == PAD] = 0
    axes[row, 0].imshow(inp[:h, :w], cmap=cmap, norm=norm)
    axes[row, 0].set_title(f'Input ({sample["task_id"][:8]})')
    axes[row, 0].axis('off')

    # Expected
    exp = sample['test_output'].numpy()
    exp[exp == PAD] = 0
    axes[row, 1].imshow(exp[:h, :w], cmap=cmap, norm=norm)
    axes[row, 1].set_title('Expected')
    axes[row, 1].axis('off')

    # Predicted
    axes[row, 2].imshow(pred[:h, :w], cmap=cmap, norm=norm)
    match = 'EXACT' if (pred[:h, :w] == exp[:h, :w]).all() else f'{(pred[:h,:w]==exp[:h,:w]).mean():.0%}'
    axes[row, 2].set_title(f'Predicted ({match})')
    axes[row, 2].axis('off')

plt.suptitle('ARC Grid Transformer v2 -- Predictions on Eval Tasks', fontsize=14)
plt.tight_layout()
plt.savefig('arc_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to arc_predictions.png')

In [ ]:
# CELL 10: Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(range(50, 50 * len(train_losses) + 1, 50), train_losses, 'b-', linewidth=1.5)
ax1.set_xlabel('Step')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

# Val accuracy curve
if val_accs:
    ax2.plot(range(200, 200 * len(val_accs) + 1, 200), [a * 100 for a in val_accs], 'g-o', linewidth=1.5)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Cell Accuracy (%)')
    ax2.set_title(f'Validation Cell Accuracy (best: {best_val_acc:.1%})')
    ax2.grid(True, alpha=0.3)

plt.suptitle('ARC Grid Transformer v2 -- Training Progress', fontsize=14)
plt.tight_layout()
plt.savefig('arc_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to arc_training_curves.png')

In [ ]:
# CELL 11: Download checkpoint
try:
    from google.colab import files
    files.download('best_arc_grid.pt')
    print('Downloading best_arc_grid.pt...')
except ImportError:
    print('Not in Colab -- checkpoint saved locally as best_arc_grid.pt')